# Climate Change & Urban Heat Island
**NB 20** | Dubai, UAE 2015→2024 | Landsat TIRS | LST analysis

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib, json
import numpy as np
import matplotlib.pyplot as plt
import pygeovision as pgv
from pygeovision.models import get_model
from pygeovision.inference.tiled import TiledInference

client = pgv.PyGeoVision()
print(client)


In [ ]:
BBOX        = (55.10, 25.00, 55.40, 25.30)
DATE_2015   = ('2015-07-01', '2015-09-30')
DATE_2024   = ('2024-07-01', '2024-09-30')
PROVIDERS   = ['planetary_computer']
BANDS_OPT   = ['B04','B05']          # Landsat 8/9 Red + NIR for NDVI
BANDS_THERM = ['B10']                # Landsat 8/9 TIRS Band 10 (10.9µm) for LST
DATA_DIR    = pathlib.Path('./results/nb20')
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
r2015 = client.search(bbox=BBOX, date_range=DATE_2015, providers=PROVIDERS,
    satellites=['Landsat-8','Landsat-9'], cloud_cover_max=10)
r2024 = client.search(bbox=BBOX, date_range=DATE_2024, providers=PROVIDERS,
    satellites=['Landsat-8','Landsat-9'], cloud_cover_max=10)
print(f"2015: {len(r2015)} | 2024: {len(r2024)}")
all_bands = BANDS_OPT + BANDS_THERM
d2015 = client.download(r2015[:1], output_dir=str(DATA_DIR/'2015'), bands=all_bands, post_process=['reproject:EPSG:32640','cog'])
d2024 = client.download(r2024[:1], output_dir=str(DATA_DIR/'2024'), bands=all_bands, post_process=['reproject:EPSG:32640','cog'])
s2015 = d2015[0].path if d2015 and d2015[0].success else None
s2024 = d2024[0].path if d2024 and d2024[0].success else None


In [ ]:
R2015 = DATA_DIR/'2015_ready.tif'; R2024 = DATA_DIR/'2024_ready.tif'
for scene, out, label in [(s2015,R2015,'2015'),(s2024,R2024,'2024')]:
    if scene:
        rep = client.validator.validate(str(scene), required_bands=3)
        r   = client.prepare_for_ai(str(scene), stack_bands=all_bands, bbox=BBOX,
            normalise='minmax', model_type='regression', output_path=str(out))
        print(f"{label}: {r['steps']}  shape={r['shape']}")


In [ ]:
# NDVI on optical bands (B04=Red, B05=NIR → bands 1,2 in stack)
import rasterio, numpy as np
for year, path in [('2015',R2015),('2024',R2024)]:
    if path.exists():
        idx_dir = DATA_DIR/'indices'; idx_dir.mkdir(exist_ok=True)
        ndvi_p = client.indices.ndvi(str(path), red_band=1, nir_band=2,
            output_path=str(idx_dir/f'ndvi_{year}.tif'))
        with rasterio.open(ndvi_p) as src: ndvi = src.read(1)
        print(f"NDVI {year}: mean={ndvi.mean():.3f}  veg>0.3: {(ndvi>0.3).mean()*100:.1f}%")


In [ ]:
# LST from TIRS Band 10 (band 3 in stack after stacking B04,B05,B10)
import rasterio, numpy as np
for year, path in [('2015',R2015),('2024',R2024)]:
    if path.exists():
        with rasterio.open(str(path)) as src:
            b10 = src.read(3).astype(np.float32)   # B10 = 3rd band
            res_m = abs(src.transform.a)
        # Landsat B10 DN to Top-of-Atmosphere radiance to brightness temperature
        # L = 0.0003342 * DN + 0.1 (typical Landsat 8 rescaling)
        # BT (K) = K2 / ln(K1/L + 1), K1=774.89, K2=1321.08 for B10
        L  = 0.0003342 * b10 + 0.1
        L  = np.where(L > 0, L, 0.001)
        BT = 1321.08 / np.log(774.89 / L + 1)   # Kelvin
        LST_C = BT - 273.15                        # Celsius
        print(f"LST {year}: mean={LST_C.mean():.1f}°C  max={LST_C.max():.1f}°C  "
              f"UHI pixels (>40°C): {(LST_C>40).mean()*100:.1f}%")


In [ ]:
# Summarise urban heat island changes
if R2015.exists() and R2024.exists():
    print("\nUrban Heat Island Analysis — Dubai 2015 vs 2024")
    print(f"  Resolution : {res_m:.0f}m × {res_m:.0f}m (Landsat)")
    print(f"  Study area : {R2024.stat().st_size/1024:.0f} KB processed scene")
    COG_OUT = DATA_DIR/'dubai_lst_cog.tif'
    client.postprocess.to_cog(str(R2024), str(COG_OUT))
    client.validator.generate_report(str(R2024), str(DATA_DIR/'climate_report.html'))


**NB20 — Climate/UHI** complete.
- Study area: Dubai UAE
- Sensor: Landsat 8/9 TIRS
- Model: LST analysis + NDVI
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG